In [0]:
%sql
create or replace temporary view pats as
With cte as (-- Unique Mapping
SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM (
                            -- INLINE TREATMENT DATA FOR 2020-2025
                            SELECT 
                                tx_data.*,
                                prov.HCO_PRIMARY_NPI,
                                prov.PRIMARY_SPECIALTY,
                                prov.SECONDARY_SPECIALTY,
                                CASE 
                                    WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
                                    WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
                                    WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                                         prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
                                    WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
                                    WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
                                    WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
                                    WHEN tx_data.npi IS NULL THEN 'NA'
                                    ELSE 'Others'
                                END AS SPECIALTY
                            FROM (
                                -- TREATMENT EVENTS FROM MEDICAL CLAIMS
                                SELECT DISTINCT 
                                    PATIENT_ID,
                                    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                                    NDC11 AS CODE,
                                    MEDICAL_EVENT_ID AS EVENT_ID,
                                    SERVICE_DATE AS FILL_DATE,
                                    PLACE_OF_SERVICE,
                                    KH_PLAN_ID AS KH_PLAN,
                                    'MEDICAL_EVENTS' AS TABLE_NAME
                                FROM com_edp_prd.com_raw.kom_medical_events 
                                WHERE NDC11 IN ('54092070001','540920700')
                                  AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'

                                UNION

                                -- TREATMENT EVENTS FROM PHARMACY CLAIMS
                                SELECT DISTINCT 
                                    PATIENT_ID,
                                    PRESCRIBER_NPI AS NPI,
                                    NDC11 AS CODE,
                                    PHARMACY_EVENT_ID as EVENT_ID,
                                    FILL_DATE,
                                    NULL AS PLACE_OF_SERVICE,
                                    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                                    'PHARMACY_EVENTS' AS TABLE_NAME
                                FROM com_edp_prd.com_raw.kom_pharmacy_events
                                WHERE NDC11 IN ('54092070001','540920700')
                                  AND TRANSACTION_RESULT = 'PAID'
                                  AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'

                                UNION

                                -- TREATMENT EVENTS FROM PROCEDURE CODES
                                SELECT DISTINCT 
                                    PATIENT_ID,
                                    RENDERING_NPI AS NPI,
                                    PROCEDURE_CODE AS CODE,   
                                    MEDICAL_EVENT_ID AS EVENT_ID,
                                    SERVICE_DATE AS FILL_DATE, 
                                    PLACE_OF_SERVICE,
                                    KH_PLAN_ID AS KH_PLAN,
                                    'MEDICAL_EVENTS' AS TABLE_NAME
                                FROM com_edp_prd.com_raw.kom_medical_events 
                                WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
                                  AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
                            ) tx_data
                            LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
                            WHERE (
                                -- PATIENTS WITH 2+ SPECIFIED DIAGNOSES
                                tx_data.PATIENT_ID IN (
                                    SELECT DISTINCT PATIENT_ID 
                                    FROM (
                                        SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                        FROM (
                                            -- SPECIFIED DIAGNOSIS CLAIMS
                                            SELECT DISTINCT 
                                                PATIENT_ID,
                                                SERVICE_DATE AS FILL_DATE
                                            FROM com_edp_prd.com_raw.kom_medical_events
                                            WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                              AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

                                            UNION

                                            SELECT DISTINCT 
                                                PATIENT_ID,
                                                FILL_DATE
                                            FROM com_edp_prd.com_raw.kom_pharmacy_events
                                            WHERE DIAGNOSIS_CODE = 'E761'
                                              AND TRANSACTION_STATUS = 'PAID'
                                              AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                                        )
                                        GROUP BY PATIENT_ID
                                    )
                                    WHERE NUMBER_OF_CLAIMS >= 2
                                )
                                OR
                                -- INCREMENTAL UNSPECIFIED PATIENTS  
                                tx_data.PATIENT_ID IN (
                                    SELECT DISTINCT PATIENT_ID 
                                    FROM (
                                        SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                                        FROM (
                                            -- UNSPECIFIED DIAGNOSIS CLAIMS
                                            SELECT DISTINCT 
                                                PATIENT_ID,
                                                SERVICE_DATE AS FILL_DATE
                                            FROM com_edp_prd.com_raw.kom_medical_events
                                            WHERE DIAGNOSIS_CODES LIKE '%E763%'
                                              AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

                                            UNION

                                            SELECT DISTINCT 
                                                PATIENT_ID,
                                                FILL_DATE
                                            FROM com_edp_prd.com_raw.kom_pharmacy_events
                                            WHERE DIAGNOSIS_CODE = 'E763'
                                              AND TRANSACTION_STATUS = 'PAID'
                                              AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                                        )
                                        GROUP BY PATIENT_ID
                                    )
                                    WHERE NUMBER_OF_CLAIMS >= 2
                                    AND PATIENT_ID IN (
                                        -- ONLY ELAPRASE TREATED UNSPECIFIED PATIENTS
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT DISTINCT PATIENT_ID
                                            FROM com_edp_prd.com_raw.kom_medical_events 
                                            WHERE NDC11 IN ('54092070001','540920700')
                                              AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'

                                            UNION

                                            SELECT DISTINCT PATIENT_ID
                                            FROM com_edp_prd.com_raw.kom_pharmacy_events
                                            WHERE NDC11 IN ('54092070001','540920700')
                                              AND TRANSACTION_RESULT = 'PAID'
                                              AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'

                                            UNION

                                            SELECT DISTINCT PATIENT_ID
                                            FROM com_edp_prd.com_raw.kom_medical_events 
                                            WHERE PROCEDURE_CODE = 'J1743'
                                              AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
                                        )
                                    )
                                    AND PATIENT_ID NOT IN (
                                        -- EXCLUDE ALREADY COUNTED SPECIFIED PATIENTS
                                        SELECT DISTINCT PATIENT_ID 
                                        FROM (
                                            SELECT DISTINCT PATIENT_ID
                                            FROM com_edp_prd.com_raw.kom_medical_events
                                            WHERE DIAGNOSIS_CODES LIKE '%E761%'
                                              AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'

                                            UNION

                                            SELECT DISTINCT PATIENT_ID
                                            FROM com_edp_prd.com_raw.kom_pharmacy_events
                                            WHERE DIAGNOSIS_CODE = 'E761'
                                              AND TRANSACTION_STATUS = 'PAID'
                                              AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                                        )
                                    )
                                )
                            )
                        )
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE, KH_PLAN, HCO_PRIMARY_NPI, PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    
)
)
SELECT * from cte;

In [0]:
%sql
-- Show ALL NPIs considered for each patient with ranking details
CREATE OR REPLACE TEMP VIEW primary_hcp_all_candidates AS
WITH base_visits AS (
    -- Aggregate to patient-NPI level FIRST
    SELECT 
        PATIENT_ID,
        NPI,
        SPECIALTY,
        HCO_PRIMARY_NPI,
        COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS,
        MAX(FILL_DATE) AS last_visit_date,
        MIN(FILL_DATE) AS first_visit_date,
        CASE 
            WHEN SPECIALTY = 'Geneticist' THEN 1 
            WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
            WHEN SPECIALTY = 'Pediatrician' THEN 3 
            WHEN SPECIALTY = 'PCP' THEN 4 
            WHEN SPECIALTY = 'NPPA' THEN 5
            WHEN SPECIALTY = 'Others' THEN 6 
            ELSE 7
        END AS PRIORITY
    FROM (
        SELECT 
            tx_data.PATIENT_ID,
            tx_data.NPI,
            tx_data.FILL_DATE,
            prov.HCO_PRIMARY_NPI,
            CASE 
                WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
                WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
                WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                     prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
                WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
                WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
                WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
                WHEN tx_data.npi IS NULL THEN 'NA'
                ELSE 'Others'
            END AS SPECIALTY
        FROM (
            SELECT DISTINCT 
                PATIENT_ID,
                COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                SERVICE_DATE AS FILL_DATE
            FROM com_edp_prd.com_raw.kom_medical_events 
            WHERE NDC11 IN ('54092070001','540920700')
              AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
            
            UNION
            
            SELECT DISTINCT 
                PATIENT_ID,
                PRESCRIBER_NPI AS NPI,
                FILL_DATE
            FROM com_edp_prd.com_raw.kom_pharmacy_events
            WHERE NDC11 IN ('54092070001','540920700')
              AND TRANSACTION_RESULT = 'PAID'
              AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
            
            UNION
            
            SELECT DISTINCT 
                PATIENT_ID,
                RENDERING_NPI AS NPI,
                SERVICE_DATE AS FILL_DATE
            FROM com_edp_prd.com_raw.kom_medical_events 
            WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
              AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        ) tx_data
        LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
        WHERE tx_data.PATIENT_ID IN (
            -- Eligible patients: 2+ E761 diagnoses in 5yr
            SELECT DISTINCT PATIENT_ID 
            FROM (
                SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
                FROM (
                    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
                    FROM com_edp_prd.com_raw.kom_medical_events
                    WHERE DIAGNOSIS_CODES LIKE '%E761%'
                      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                    UNION
                    SELECT DISTINCT PATIENT_ID, FILL_DATE
                    FROM com_edp_prd.com_raw.kom_pharmacy_events
                    WHERE DIAGNOSIS_CODE = 'E761'
                      AND TRANSACTION_STATUS = 'PAID'
                      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                )
                GROUP BY PATIENT_ID
            )
            WHERE NUMBER_OF_CLAIMS >= 2
        )
    )
    GROUP BY PATIENT_ID, NPI, SPECIALTY, HCO_PRIMARY_NPI
),
ranked AS (
    SELECT 
        *,
        RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, last_visit_date DESC, NPI) AS hcp_rank
    FROM base_visits
)
SELECT 
    PATIENT_ID,
    NPI,
    SPECIALTY,
    PRIORITY,
    NO_OF_VISITS,
    last_visit_date,
    first_visit_date,
    hcp_rank,
    HCO_PRIMARY_NPI,
    CASE 
        WHEN hcp_rank = 1 THEN 'PRIMARY HCP - SELECTED'
        ELSE 'Not selected'
    END AS status,
    CASE 
        WHEN hcp_rank = 1 THEN 
            CASE 
                WHEN PRIORITY = 1 THEN 'Reason: Geneticist (highest specialty priority)'
                WHEN PRIORITY = 2 THEN 'Reason: Psychiatry & Neurology (2nd highest specialty)'
                WHEN PRIORITY = 3 THEN 'Reason: Pediatrician (3rd highest specialty)'
                WHEN PRIORITY = 4 THEN 'Reason: PCP (4th highest specialty)'
                WHEN PRIORITY = 5 THEN 'Reason: NPPA with most visits (' || CAST(NO_OF_VISITS AS STRING) || ' visits)'
                WHEN PRIORITY = 6 THEN 'Reason: Other specialty with most visits (' || CAST(NO_OF_VISITS AS STRING) || ' visits)'
            END
        ELSE 
            CASE 
                WHEN PRIORITY > (SELECT MIN(PRIORITY) FROM ranked r2 WHERE r2.PATIENT_ID = ranked.PATIENT_ID AND r2.hcp_rank = 1) 
                    THEN 'Not selected: Lower specialty priority'
                WHEN NO_OF_VISITS < (SELECT MAX(NO_OF_VISITS) FROM ranked r2 WHERE r2.PATIENT_ID = ranked.PATIENT_ID AND r2.PRIORITY = ranked.PRIORITY)
                    THEN 'Not selected: Fewer visits than other ' || SPECIALTY
                WHEN last_visit_date < (SELECT MAX(last_visit_date) FROM ranked r2 WHERE r2.PATIENT_ID = ranked.PATIENT_ID AND r2.PRIORITY = ranked.PRIORITY AND r2.NO_OF_VISITS = ranked.NO_OF_VISITS)
                    THEN 'Not selected: Older last visit date'
                ELSE 'Not selected: Higher NPI (tiebreaker)'
            END
    END AS reason
FROM ranked
ORDER BY PATIENT_ID, hcp_rank;

-- To see just patient 5HREMYP4:
SELECT * FROM primary_hcp_all_candidates WHERE PATIENT_ID = '5HREMYP4';

In [0]:
%sql
select distinct n_pats, NPI,specialty from pats

In [0]:
%sql
select * from com_edp_prd.cmpa_insights_internal_schema.patient360

In [0]:
%sql
-- =============================================================================
-- PRIMARY HCP PATIENT-LEVEL BREAKDOWN
-- =============================================================================
-- Shows ALL NPIs considered for each patient with full ranking details
-- Treatment Window: Aug 2023 - Jul 2025 (2 years)
-- Diagnosis Window: Aug 2020 - Jul 2025 (5 years)
-- =============================================================================

WITH treatment_with_specialty AS (
    -- Get all treatment claims with specialty classification
    SELECT 
        tx_data.PATIENT_ID,
        tx_data.NPI,
        tx_data.FILL_DATE,
        prov.PRIMARY_SPECIALTY as raw_specialty,
        prov.HCO_PRIMARY_NPI,
        CASE 
            WHEN prov.primary_specialty LIKE '%Genetic%' OR prov.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
            WHEN prov.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
            WHEN prov.primary_specialty LIKE '%Psychiatry & Neurology%' OR prov.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                 prov.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
            WHEN prov.primary_specialty LIKE '%Nurse Practitioner%' OR prov.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
            WHEN prov.primary_specialty LIKE '%Internal Medicine%' OR prov.secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
            WHEN prov.primary_specialty LIKE '%Family Medicine%' OR prov.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
            WHEN tx_data.npi IS NULL THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY
    FROM (
        -- Treatment events - Medical NDC
        SELECT DISTINCT 
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events 
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        
        UNION
        
        -- Treatment events - Pharmacy NDC
        SELECT DISTINCT 
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        
        UNION
        
        -- Treatment events - Procedure codes
        SELECT DISTINCT 
            PATIENT_ID,
            RENDERING_NPI AS NPI,
            SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events 
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
    ) tx_data
    LEFT JOIN com_edp_prd.com_raw.kom_providers prov ON tx_data.npi = prov.npi
    WHERE tx_data.PATIENT_ID IN (
        -- Eligible patients: 2+ E761 diagnoses in 5yr
        SELECT DISTINCT PATIENT_ID 
        FROM (
            SELECT PATIENT_ID, COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
            FROM (
                SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
                FROM com_edp_prd.com_raw.kom_medical_events
                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                UNION
                SELECT DISTINCT PATIENT_ID, FILL_DATE
                FROM com_edp_prd.com_raw.kom_pharmacy_events
                WHERE DIAGNOSIS_CODE = 'E761'
                  AND TRANSACTION_STATUS = 'PAID'
                  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
            )
            GROUP BY PATIENT_ID
        )
        WHERE NUMBER_OF_CLAIMS >= 2
    )
),
patient_npi_summary AS (
    -- Aggregate to ONE ROW per patient-NPI
    SELECT 
        PATIENT_ID,
        NPI,
        SPECIALTY,
        raw_specialty,
        HCO_PRIMARY_NPI,
        COUNT(DISTINCT FILL_DATE) AS no_of_visits,
        MAX(FILL_DATE) AS last_visit_date,
        MIN(FILL_DATE) AS first_visit_date,
        CASE 
            WHEN SPECIALTY = 'Geneticist' THEN 1 
            WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
            WHEN SPECIALTY = 'Pediatrician' THEN 3 
            WHEN SPECIALTY = 'PCP' THEN 4 
            WHEN SPECIALTY = 'NPPA' THEN 5
            WHEN SPECIALTY = 'Others' THEN 6 
            ELSE 7
        END AS specialty_priority
    FROM treatment_with_specialty
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI, SPECIALTY, raw_specialty, HCO_PRIMARY_NPI
),
ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY specialty_priority ASC, no_of_visits DESC, last_visit_date DESC, NPI ASC
        ) AS hcp_rank
    FROM patient_npi_summary
)
SELECT 
    PATIENT_ID,
    NPI,
    SPECIALTY as specialty_category,
    raw_specialty as provider_specialty,
    specialty_priority,
    no_of_visits,
    last_visit_date,
    first_visit_date,
    hcp_rank,
    HCO_PRIMARY_NPI,
    CASE 
        WHEN hcp_rank = 1 THEN 'YES - PRIMARY HCP'
        ELSE 'No'
    END AS is_primary_hcp,
    CASE 
        WHEN hcp_rank = 1 THEN 
            'Selected: Priority=' || CAST(specialty_priority AS STRING) || 
            ', Visits=' || CAST(no_of_visits AS STRING) || 
            ', Last Visit=' || CAST(last_visit_date AS STRING)
        ELSE 
            'Not selected: See lower rank'
    END AS selection_reason
FROM ranked_hcps
ORDER BY PATIENT_ID, hcp_rank;

-- Filter for specific patient
-- SELECT * FROM primary_hcp_all_candidates WHERE PATIENT_ID = '5HREMYP4';

In [0]:
%sql
select 
from 